# Assignment: Building and Comparing Language Models

**Objective:** In this assignment, you have to implement RNN language models, evaluate their performance using the perplexity metric, and compare their result with Neural language model of fixed window size and Probabilistic language Model.

1.  **A Neural RNN-based Language Model** - A modern deep learning approach.
---

## What is Perplexity?

Perplexity is the standard evaluation metric for language models. It measures how well a probability model predicts a sample. A lower perplexity indicates that the model is better at predicting the text.

It is calculated as the exponentiated average negative log-likelihood of a sequence of words $W = (w_1, w_2, ..., w_N)$:

$$ Perplexity(W) = P(w_1, w_2, ..., w_N)^{-\frac{1}{N}} = \sqrt[N]{\frac{1}{P(w_1, w_2, ..., w_N)}} $$

For our n-gram and RNN models, we will compute this on a held-out test set.

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import collections
import re

## Data Preparation

We have to use below corpus for this assignment for training . We'll also split it into a training set and a testing set.

In [3]:
corpus = """
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)

"The height of his glory"--that was what the women called it. I can hear Mrs. Gideon Thwing--his last Chicago sitter--deploring his unaccountable abdication. "Of course it's going to send the value of my picture 'way up; but I don't think of that, Mr. Rickham--the loss to Arrt is all I think of." The word, on Mrs. Thwing's lips, multiplied its _rs_ as though they were reflected in an endless vista of mirrors. And it was not only the Mrs. Thwings who mourned. Had not the exquisite Hermia Croft, at the last Grafton Gallery show, stopped me before Gisburn's "Moon-dancers" to say, with tears in her eyes: "We shall not look upon its like again"?

Well!--even through the prism of Hermia's tears I felt able to face the fact with equanimity. Poor Jack Gisburn! The women had made him--it was fitting that they should mourn him. Among his own sex fewer regrets were heard, and in his own trade hardly a murmur. Professional jealousy? Perhaps. If it were, the honour of the craft was vindicated by little Claude Nutley, who, in all good faith, brought out in the Burlington a very handsome "obituary" on Jack--one of those showy articles stocked with random technicalities that I have heard (I won't say by whom) compared to Gisburn's painting. And so--his resolve being apparently irrevocable--the discussion gradually died out, and, as Mrs. Thwing had predicted, the price of "Gisburns" went up.

It was not till three years later that, in the course of a few weeks' idling on the Riviera, it suddenly occurred to me to wonder why Gisburn had given up his painting. On reflection, it really was a tempting problem. To accuse his wife would have been too easy--his fair sitters had been denied the solace of saying that Mrs. Gisburn had "dragged him down." For Mrs. Gisburn--as such--had not existed till nearly a year after Jack's resolve had been taken. It might be that he had married her--since he liked his ease--because he didn't want to go on painting; but it would have been hard to prove that he had given up his painting because he had married her.

Of course, if she had not dragged him down, she had equally, as Miss Croft contended, failed to "lift him up"--she had not led him back to the easel. To put the brush into his hand again--what a vocation for a wife! But Mrs. Gisburn appeared to have disdained it--and I felt it might be interesting to find out why.

The desultory life of the Riviera lends itself to such purely academic speculations; and having, on my way to Monte Carlo, caught a glimpse of Jack's balustraded terraces between the pines, I had myself borne thither the next day.

I found the couple at tea beneath their palm-trees; and Mrs. Gisburn's welcome was so genial that, in the ensuing weeks, I claimed it frequently. It was not that my hostess was "interesting": on that point I could have given Miss Croft the fullest reassurance. It was just because she was _not_ interesting--if I may be pardoned the bull--that I found her so. For Jack, all his life, had been surrounded by interesting women: they had fostered his art, it had been reared in the hot-house of their adulation. And it was therefore instructive to note what effect the "deadening atmosphere of mediocrity" (I quote Miss Croft) was having on him.

I have mentioned that Mrs. Gisburn was rich; and it was immediately perceptible that her husband was extracting from this circumstance a delicate but substantial satisfaction. It is, as a rule, the people who scorn money who get most out of it; and Jack's elegant disdain of his wife's big balance enabled him, with an appearance of perfect good-breeding, to transmute it into objects of art and luxury. To the latter, I must add, he remained relatively indifferent; but he was buying Renaissance bronzes and eighteenth-century pictures with a discrimination that bespoke the amplest resources.

"Money's only excuse is to put beauty into circulation," was one of the axioms he laid down across the Sevres and silver of an exquisitely appointed luncheon-table, when, on a later day, I had again run over from Monte Carlo; and Mrs. Gisburn, beaming on him, added for my enlightenment: "Jack is so morbidly sensitive to every form of beauty."

Poor Jack! It had always been his fate to have women say such things of him: the fact should be set down in extenuation. What struck me now was that, for the first time, he resented the tone. I had seen him, so often, basking under similar tributes--was it the conjugal note that robbed them of their savour? No--for, oddly enough, it became apparent that he was fond of Mrs. Gisburn--fond enough not to see her absurdity. It was his own absurdity he seemed to be wincing under--his own attitude as an object for garlands and incense.

"My dear, since I've chucked painting people don't say that stuff about me--they say it about Victor Grindle," was his only protest, as he rose from the table and strolled out onto the sunlit terrace.

I glanced after him, struck by his last word. Victor Grindle was, in fact, becoming the man of the moment--as Jack himself, one might put it, had been the man of the hour. The younger artist was said to have formed himself at my friend's feet, and I wondered if a tinge of jealousy underlay the latter's mysterious abdication. But no--for it was not till after that event that the _rose Dubarry_ drawing-rooms had begun to display their "Grindles."

I turned to Mrs. Gisburn, who had lingered to give a lump of sugar to her spaniel in the dining-room.

"Why _has_ he chucked painting?" I asked abruptly.

She raised her eyebrows with a hint of good-humoured surprise.

"Oh, he doesn't _have_ to now, you know; and I want him to enjoy himself," she said quite simply.

I looked about the spacious white-panelled room, with its _famille-verte_ vases repeating the tones of the pale damask curtains, and its eighteenth-century pastels in delicate faded frames.

"Has he chucked his pictures too? I haven't seen a single one in the house."

A slight shade of constraint crossed Mrs. Gisburn's open countenance. "It's his ridiculous modesty, you know. He says they're not fit to have about; he's sent them all away except one--my portrait--and that I have to keep upstairs."

His ridiculous modesty--Jack's modesty about his pictures? My curiosity was growing like the bean-stalk. I said persuasively to my hostess: "I must really see your portrait, you know."

She glanced out almost timorously at the terrace where her husband, lounging in a hooded chair, had lit a cigar and drawn the Russian deerhound's head between his knees.

"Well, come while he's not looking," she said, with a laugh that tried to hide her nervousness; and I followed her between the marble Emperors of the hall, and up the wide stairs with terra-cotta nymphs poised among flowers at each landing.

In the dimmest corner of her boudoir, amid a profusion of delicate and distinguished objects, hung one of the familiar oval canvases, in the inevitable garlanded frame. The mere outline of the frame called up all Gisburn's past!

Mrs. Gisburn drew back the window-curtains, moved aside a _jardiniere_ full of pink azaleas, pushed an arm-chair away, and said: "If you stand here you can just manage to see it. I had it over the mantel-piece, but he wouldn't let it stay."

Yes--I could just manage to see it--the first portrait of Jack's I had ever had to strain my eyes over! Usually they had the place of honour--say the central panel in a pale yellow or _rose Dubarry_ drawing-room, or a monumental easel placed so that it took the light through curtains of old Venetian point. The more modest place became the picture better; yet, as my eyes grew accustomed to the half-light, all the characteristic qualities came out--all the hesitations disguised as audacities, the tricks of prestidigitation by which, with such consummate skill, he managed to divert attention from the real business of the picture to some pretty irrelevance of detail. Mrs. Gisburn, presenting a neutral surface to work on--forming, as it were, so inevitably the background of her own picture--had lent herself in an unusual degree to the display of this false virtuosity. The picture was one of Jack's "strongest," as his admirers would have put it--it represented, on his part, a swelling of muscles, a congesting of veins, a balancing, straddling and straining, that reminded one of the circus-clown's ironic efforts to lift a feather. It met, in short, at every point the demand of lovely woman to be painted "strongly" because she was tired of being painted "sweetly"--and yet not to lose an atom of the sweetness.

"It's the last he painted, you know," Mrs. Gisburn said with pardonable pride. "The last but one," she corrected herself--"but the other doesn't count, because he destroyed it."

"Destroyed it?" I was about to follow up this clue when I heard a footstep and saw Jack himself on the threshold.

As he stood there, his hands in the pockets of his velveteen coat, the thin brown waves of hair pushed back from his white forehead, his lean sunburnt cheeks furrowed by a smile that lifted the tips of a self-confident moustache, I felt to what a degree he had the same quality as his pictures--the quality of looking cleverer than he was.

His wife glanced at him deprecatingly, but his eyes travelled past her to the portrait.

"Mr. Rickham wanted to see it," she began, as if excusing herself. He shrugged his shoulders, still smiling.

"Oh, Rickham found me out long ago," he said lightly; then, passing his arm through mine: "Come and see the rest of the house."

He showed it to me with a kind of naive suburban pride: the bath-rooms, the speaking-tubes, the dress-closets, the trouser-presses--all the complex simplifications of the millionaire's domestic economy. And whenever my wonder paid the expected tribute he said, throwing out his chest a little: "Yes, I really don't see how people manage to live without that."

Well--it was just the end one might have foreseen for him. Only he was, through it all and in spite of it all--as he had been through, and in spite of, his pictures--so handsome, so charming, so disarming, that one longed to cry out: "Be dissatisfied with your leisure!" as once one had longed to say: "Be dissatisfied with your work!"

But, with the cry on my lips, my diagnosis suffered an unexpected check.

"This is my own lair," he said, leading me into a dark plain room at the end of the florid vista. It was square and brown and leathery: no "effects"; no bric-a-brac, none of the air of posing for reproduction in a picture weekly--above all, no least sign of ever having been used as a studio.

The fact brought home to me the absolute finality of Jack's break with his old life.

"Don't you ever dabble with paint any more?" I asked, still looking about for a trace of such activity.

"Never," he said briefly.

"Or water-colour--or etching?"

His confident eyes grew dim, and his cheeks paled a little under their handsome sunburn.

"Never think of it, my dear fellow--any more than if I'd never touched a brush."

And his tone told me in a flash that he never thought of anything else.

I moved away, instinctively embarrassed by my unexpected discovery; and as I turned, my eye fell on a small picture above the mantel-piece--the only object breaking the plain oak panelling of the room.

"Oh, by Jove!" I said.

It was a sketch of a donkey--an old tired donkey, standing in the rain under a wall.

"By Jove--a Stroud!" I cried.

He was silent; but I felt him close behind me, breathing a little quickly.

"What a wonder! Made with a dozen lines--but on everlasting foundations. You lucky chap, where did you get it?"

He answered slowly: "Mrs. Stroud gave it to me."

"Ah--I didn't know you even knew the Strouds. He was such an inflexible hermit."

"I didn't--till after. . . . She sent for me to paint him when he was dead."

"When he was dead? You?"

I must have let a little too much amazement escape through my surprise, for he answered with a deprecating laugh: "Yes--she's an awful simpleton, you know, Mrs. Stroud. Her only idea was to have him done by a fashionable painter--ah, poor Stroud! She thought it the surest way of proclaiming his greatness--of forcing it on a purblind public. And at the moment I was _the_ fashionable painter."

"Ah, poor Stroud--as you say. Was _that_ his history?"

"That was his history. She believed in him, gloried in him--or thought she did. But she couldn't bear not to have all the drawing-rooms with her. She couldn't bear the fact that, on varnishing days, one could always get near enough to see his pictures. Poor woman! She's just a fragment groping for other fragments. Stroud is the only whole I ever knew."

"You ever knew? But you just said--"

Gisburn had a curious smile in his eyes.

"Oh, I knew him, and he knew me--only it happened after he was dead."

I dropped my voice instinctively. "When she sent for you?"

"Yes--quite insensible to the irony. She wanted him vindicated--and by me!"

He laughed again, and threw back his head to look up at the sketch of the donkey. "There were days when I couldn't look at that thing--couldn't face it. But I forced myself to put it here; and now it's cured me--cured me. That's the reason why I don't dabble any more, my dear Rickham; or rather Stroud himself is the reason."

For the first time my idle curiosity about my companion turned into a serious desire to understand him better.

"I wish you'd tell me how it happened," I said.

He stood looking up at the sketch, and twirling between his fingers a cigarette he had forgotten to light. Suddenly he turned toward me.

"I'd rather like to tell you--because I've always suspected you of loathing my work."

I made a deprecating gesture, which he negatived with a good-humoured shrug.

"Oh, I didn't care a straw when I believed in myself--and now it's an added tie between us!"

He laughed slightly, without bitterness, and pushed one of the deep arm-chairs forward. "There: make yourself comfortable--and here are the cigars you like."

He placed them at my elbow and continued to wander up and down the room, stopping now and then beneath the picture.

"How it happened? I can tell you in five minutes--and it didn't take much longer to happen. . . . I can remember now how surprised and pleased I was when I got Mrs. Stroud's note. Of course, deep down, I had always _felt_ there was no one like him--only I had gone with the stream, echoed the usual platitudes about him, till I half got to think he was a failure, one of the kind that are left behind. By Jove, and he _was_ left behind--because he had come to stay! The rest of us had to let ourselves be swept along or go under, but he was high above the current--on everlasting foundations, as you say.

"Well, I went off to the house in my most egregious mood--rather moved, Lord forgive me, at the pathos of poor Stroud's career of failure being crowned by the glory of my painting him! Of course I meant to do the picture for nothing--I told Mrs. Stroud so when she began to stammer something about her poverty. I remember getting off a prodigious phrase about the honour being _mine_--oh, I was princely, my dear Rickham! I was posing to myself like one of my own sitters.

"Then I was taken up and left alone with him. I had sent all my traps in advance, and I had only to set up the easel and get to work. He had been dead only twenty-four hours, and he died suddenly, of heart disease, so that there had been no preliminary work of destruction--his face was clear and untouched. I had met him once or twice, years before, and thought him insignificant and dingy. Now I saw that he was superb.

"I was glad at first, with a merely aesthetic satisfaction: glad to have my hand on such a 'subject.' Then his strange life-likeness began to affect me queerly--as I blocked the head in I felt as if he were watching me do it. The sensation was followed by the thought: if he _were_ watching me, what would he say to my way of working? My strokes began to go a little wild--I felt nervous and uncertain.

"Once, when I looked up, I seemed to see a smile behind his close grayish beard--as if he had the secret, and were amusing himself by holding it back from me. That exasperated me still more. The secret? Why, I had a secret worth twenty of his! I dashed at the canvas furiously, and tried some of my bravura tricks. But they failed me, they crumbled. I saw that he wasn't watching the showy bits--I couldn't distract his attention; he just kept his eyes on the hard passages between. Those were the ones I had always shirked, or covered up with some lying paint. And how he saw through my lies!

"I looked up again, and caught sight of that sketch of the donkey hanging on the wall near his bed. His wife told me afterward it was the last thing he had done--just a note taken with a shaking hand, when he was down in Devonshire recovering from a previous heart attack. Just a note! But it tells his whole history. There are years of patient scornful persistence in every line. A man who had swum with the current could never have learned that mighty up-stream stroke. . . .

"I turned back to my work, and went on groping and muddling; then I looked at the donkey again. I saw that, when Stroud laid in the first stroke, he knew just what the end would be. He had possessed his subject, absorbed it, recreated it. When had I done that with any of my things? They hadn't been born of me--I had just adopted them. . . .

"Hang it, Rickham, with that face watching me I couldn't do another stroke. The plain truth was, I didn't know where to put it--_I had never known_. Only, with my sitters and my public, a showy splash of colour covered up the fact--I just threw paint into their faces. . . . Well, paint was the one medium those dead eyes could see through--see straight to the tottering foundations underneath. Don't you know how, in talking a foreign language, even fluently, one says half the time not what one wants to but what one can? Well--that was the way I painted; and as he lay there and watched me, the thing they called my 'technique' collapsed like a house of cards. He didn't sneer, you understand, poor Stroud--he just lay there quietly watching, and on his lips, through the gray beard, I seemed to hear the question: 'Are you sure you know where you're coming out?'

"If I could have painted that face, with that question on it, I should have done a great thing. The next greatest thing was to see that I couldn't--and that grace was given me. But, oh, at that minute, Rickham, was there anything on earth I wouldn't have given to have Stroud alive before me, and to hear him say: 'It's not too late--I'll show you how'?

"It _was_ too late--it would have been, even if he'd been alive. I packed up my traps, and went down and told Mrs. Stroud. Of course I didn't tell her _that_--it would have been Greek to her. I simply said I couldn't paint him, that I was too moved. She rather liked the idea--she's so romantic! It was that that made her give me the donkey. But she was terribly upset at not getting the portrait--she did so want him 'done' by some one showy! At first I was afraid she wouldn't let me off--and at my wits' end I suggested Grindle. Yes, it was I who started Grindle: I told Mrs. Stroud he was the 'coming' man, and she told somebody else, and so it got to be true. . . . And he painted Stroud without wincing; and she hung the picture among her husband's things. . . ."

He flung himself down in the arm-chair near mine, laid back his head, and clasping his arms beneath it, looked up at the picture above the chimney-piece.

"I like to fancy that Stroud himself would have given it to me, if he'd been able to say what he thought that day."

And, in answer to a question I put half-mechanically--"Begin again?" he flashed out. "When the one thing that brings me anywhere near him is that I knew enough to leave off?"

He stood up and laid his hand on my shoulder with a laugh. "Only the irony of it is that I _am_ still painting--since Grindle's doing it for me! The Strouds stand alone, and happen once--but there's no exterminating our kind of art."
"""

# Part 1: Neural RNN Language Model

Now, you have to build an RNN to perform the same task. The RNN will process a sequence of words and predict the next one.

### Step 1.1: Create Sequential Data

Unlike the n-gram model, which looks at pairs of words, the RNN can look at a longer history. We need to create input sequences and corresponding targets.

In [4]:
# Basic preprocessing
clean_text = re.sub(r"[^a-zA-Z\s']", " ", corpus.lower())
tokens = clean_text.split()

# Train/test split (word-level)
split_idx = int(0.8 * len(tokens))
train_tokens = tokens[:split_idx]
test_tokens = tokens[split_idx:]

# Build vocabulary from train set and add UNK token for unseen test words
word_counts = collections.Counter(train_tokens)
vocab = ["<unk>"] + sorted(word_counts.keys())
word_to_idx = {word: idx for idx, word in enumerate(vocab)}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}
unk_idx = word_to_idx["<unk>"]

def encode(token_list):
    return [word_to_idx.get(token, unk_idx) for token in token_list]

train_ids = encode(train_tokens)
test_ids = encode(test_tokens)

seq_len = 5

def make_sequences(token_ids, sequence_length):
    x, y = [], []
    for i in range(len(token_ids) - sequence_length):
        x.append(token_ids[i:i + sequence_length])
        y.append(token_ids[i + sequence_length])
    return x, y

train_X, train_y = make_sequences(train_ids, seq_len)
test_X, test_y = make_sequences(test_ids, seq_len)

train_X = torch.tensor(train_X, dtype=torch.long)
train_y = torch.tensor(train_y, dtype=torch.long)
test_X = torch.tensor(test_X, dtype=torch.long)
test_y = torch.tensor(test_y, dtype=torch.long)

print(f"Total tokens: {len(tokens)}")
print(f"Train tokens: {len(train_tokens)}, Test tokens: {len(test_tokens)}")
print(f"Vocab size: {len(vocab)}")
print(f"Train sequences: {len(train_X)}, Test sequences: {len(test_X)}")

Total tokens: 3749
Train tokens: 2999, Test tokens: 750
Vocab size: 985
Train sequences: 2994, Test sequences: 745


### Step 1.2: Define the RNN Model

Now, define model. It should consist of:
1.  An **Embedding layer** to convert word indices into dense vectors.
2.  An **RNN layer** to process the sequence.
3.  A **Linear (fully-connected) layer** to map the RNN output to vocabulary scores.

In [5]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embedding(x)
        out, hidden = self.rnn(emb, hidden)
        logits = self.fc(out[:, -1, :])
        return logits, hidden

embedding_dim = 64
hidden_dim = 128

model = RNNLanguageModel(
    vocab_size=len(vocab),
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(model)
print(f"Running on: {device}")

RNNLanguageModel(
  (embedding): Embedding(985, 64)
  (rnn): RNN(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=985, bias=True)
)
Running on: cpu


### Step 1.3: Train the RNN Model

Complete the training loop below. For each input sequence, you need to perform a forward pass, calculate the loss, and update the model's weights.

In [6]:
epochs = 40
batch_size = 64

if len(train_X) == 0:
    raise ValueError("Training data is empty. Check preprocessing and sequence length.")

for epoch in range(1, epochs + 1):
    model.train()
    permutation = torch.randperm(len(train_X))

    epoch_loss = 0.0

    for i in range(0, len(train_X), batch_size):
        indices = permutation[i:i + batch_size]
        x_batch = train_X[indices].to(device)
        y_batch = train_y[indices].to(device)

        optimizer.zero_grad()
        logits, _ = model(x_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        epoch_loss += loss.item() * x_batch.size(0)

    avg_loss = epoch_loss / len(train_X)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d}/{epochs} - Train Loss: {avg_loss:.4f}")

Epoch 01/40 - Train Loss: 6.6501
Epoch 05/40 - Train Loss: 4.8830
Epoch 10/40 - Train Loss: 3.4957
Epoch 15/40 - Train Loss: 2.2741
Epoch 20/40 - Train Loss: 1.3252
Epoch 25/40 - Train Loss: 0.6658
Epoch 30/40 - Train Loss: 0.2927
Epoch 35/40 - Train Loss: 0.1505
Epoch 40/40 - Train Loss: 0.0919


### Step 1.4: Calculate Perplexity for the RNN Model

Now, write a function to calculate the perplexity of your trained RNN on the test set.

In [7]:
def calculate_perplexity(model, X, y, criterion, batch_size=128):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x_batch = X[i:i + batch_size].to(device)
            y_batch = y[i:i + batch_size].to(device)

            logits, _ = model(x_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * x_batch.size(0)
            total_tokens += x_batch.size(0)

    avg_nll = total_loss / total_tokens
    return float(np.exp(avg_nll))

rnn_perplexity = calculate_perplexity(model, test_X, test_y, criterion)
print(f"RNN Model Perplexity on Test Set: {rnn_perplexity:.4f}")

RNN Model Perplexity on Test Set: 3419.4075


In [8]:
# Part 2 helpers: compute baseline perplexities for comparison

# ---------- 1) Probabilistic bigram LM (add-one smoothing) ----------
def bigram_perplexity(train_ids, test_ids, vocab_size):
    bigram_counts = collections.Counter()
    prev_counts = collections.Counter()

    for i in range(1, len(train_ids)):
        prev_w = train_ids[i - 1]
        curr_w = train_ids[i]
        bigram_counts[(prev_w, curr_w)] += 1
        prev_counts[prev_w] += 1

    log_prob_sum = 0.0
    n = 0

    for i in range(1, len(test_ids)):
        prev_w = test_ids[i - 1]
        curr_w = test_ids[i]
        prob = (bigram_counts[(prev_w, curr_w)] + 1.0) / (prev_counts[prev_w] + vocab_size)
        log_prob_sum += np.log(prob)
        n += 1

    return float(np.exp(-log_prob_sum / n))

ngram_perplexity = bigram_perplexity(train_ids, test_ids, len(vocab))


# ---------- 2) Neural fixed-window LM ----------
class FixedWindowNNLM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, context_size, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.fc1 = nn.Linear(context_size * embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)                 # [batch, context, emb_dim]
        emb_flat = emb.view(emb.size(0), -1)   # [batch, context*emb_dim]
        h = torch.relu(self.fc1(emb_flat))
        logits = self.fc2(h)
        return logits

context_size = seq_len
ff_model = FixedWindowNNLM(
    vocab_size=len(vocab),
    embedding_dim=64,
    context_size=context_size,
    hidden_dim=128,
).to(device)

ff_criterion = nn.CrossEntropyLoss()
ff_optimizer = optim.Adam(ff_model.parameters(), lr=0.001)

ff_epochs = 25
ff_batch_size = 64

for epoch in range(1, ff_epochs + 1):
    ff_model.train()
    permutation = torch.randperm(len(train_X))
    epoch_loss = 0.0

    for i in range(0, len(train_X), ff_batch_size):
        indices = permutation[i:i + ff_batch_size]
        x_batch = train_X[indices].to(device)
        y_batch = train_y[indices].to(device)

        ff_optimizer.zero_grad()
        logits = ff_model(x_batch)
        loss = ff_criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(ff_model.parameters(), max_norm=1.0)
        ff_optimizer.step()

        epoch_loss += loss.item() * x_batch.size(0)


def neural_perplexity(model, X, y, criterion, batch_size=128):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            x_batch = X[i:i + batch_size].to(device)
            y_batch = y[i:i + batch_size].to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            total_loss += loss.item() * x_batch.size(0)
            total_tokens += x_batch.size(0)

    avg_nll = total_loss / total_tokens
    return float(np.exp(avg_nll))

fixed_window_perplexity = neural_perplexity(ff_model, test_X, test_y, ff_criterion)

print(f"N-gram (bigram) Perplexity on Test Set: {ngram_perplexity:.4f}")
print(f"Neural Fixed-Window Perplexity on Test Set: {fixed_window_perplexity:.4f}")
print(f"RNN Perplexity on Test Set: {rnn_perplexity:.4f}")

N-gram (bigram) Perplexity on Test Set: 758.6031
Neural Fixed-Window Perplexity on Test Set: 38629.3234
RNN Perplexity on Test Set: 3419.4075


# Part 2: Comparison and Analysis

After implementing and running all three models, answer the following questions.

### Results

* **N-gram Model Perplexity:** 758.6031
* **Neural model of fixed window size Perplexity:** 38629.3234
* **RNN Model Perplexity:** 3419.4075

### Analysis Questions

1.  **Which model achieved a lower perplexity on the test set? What does this indicate?**

    The **n-gram model** achieved the lowest perplexity in this experiment. This indicates that, for this specific dataset size and setup, the smoothed bigram model generalized better than the neural models. A lower perplexity means the model assigned higher probability to the actual test sequence.

2.  **What are the main advantages of using an RNN language model over a statistical n-gram model and neural language model of fixed window size?**

    RNNs can model **longer context dependencies** because they carry hidden state across sequence positions, while n-gram and fixed-window models only use limited context. RNNs also learn **distributed representations** (embeddings + sequence dynamics), which can capture semantic and syntactic patterns more flexibly than count-based methods. Compared to fixed-window neural models, RNNs are better suited for variable-length history and can represent information beyond a fixed context window.